# Build Dataset: DJF ENSO Vertical Cross-Section Correlations

Build cached DJF Niño3.4 correlation cross-sections for the Maritime Continent sector.

This notebook only computes and saves NetCDF caches. Plotting is handled separately.

# 1. Import Library

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

PROJECT_ROOT = Path('/Users/rizzie/TugasAkhir/data_processing')
ERA5_ROOT = PROJECT_ROOT / 'external/ClimateData/era5-monthly'
INDEX_ROOT = PROJECT_ROOT / 'external/ClimateData/index-monthly'
CACHE_DIR = PROJECT_ROOT / 'data/intermediate/divided_correlation'

Q_PATH = ERA5_ROOT / 'specific_humidity_1980-2025_1000hpa-100hpa.nc'
UV_PATH = ERA5_ROOT / 'uv_wind_1980-2025_1000hpa-100hpa.nc'
W_PATH = ERA5_ROOT / 'vertical_velocity_1980-2025_1000hpa-100hpa.nc'
NINO34_PATH = INDEX_ROOT / 'nino34.anom.csv'

LON_CACHE_PATH = CACHE_DIR / 'corr_xsec_lon_pressure_q_u_omega.nc'
LAT_CACHE_PATH = CACHE_DIR / 'corr_xsec_lat_pressure_q_v_omega.nc'

DJF_YEARS = np.arange(1981, 2026)
P1_YEARS = np.arange(1981, 2007)
P2_YEARS = np.arange(2007, 2026)
TIME_START = '1980-12-01'
TIME_END = '2025-02-28'
LON_BAND = slice(95.0, 125.0)
LAT_BAND = slice(-10.0, 5.0)

# Keep True while validating the corrected domain logic. Set back to False only after caches are confirmed.
OVERWRITE = True


def pick_data_var(ds: xr.Dataset, target: str) -> xr.DataArray:
    target_key = target.replace('_', '').lower()
    for name in ds.data_vars:
        normalized = name.replace('_', '').lower()
        if normalized == target_key:
            return ds[name]
    for name in ds.data_vars:
        normalized = name.replace('_', '').lower()
        if target_key in normalized:
            return ds[name]
    raise KeyError(f'Could not find variable matching {target!r} in {list(ds.data_vars)}')


def standardize_da(da: xr.DataArray) -> xr.DataArray:
    if 'valid_time' in da.coords:
        da = da.drop_vars('valid_time')

    rename_map: dict[str, str] = {}
    for old, new in (
        ('latitude', 'lat'),
        ('longitude', 'lon'),
        ('pressure_level', 'level'),
        ('isobaricInhPa', 'level'),
        ('isobaricInPa', 'level'),
    ):
        if old in da.dims or old in da.coords:
            rename_map[old] = new

    if rename_map:
        da = da.rename(rename_map)

    if 'lon' in da.coords:
        da = da.assign_coords(lon=(da['lon'] % 360)).sortby('lon')
    if 'lat' in da.coords:
        da = da.sortby('lat')
    if 'level' in da.coords:
        da = da.sortby('level')

    return da


def load_era5_field(path: Path, target: str) -> xr.DataArray:
    ds = xr.open_dataset(
        path,
        chunks={'time': 12, 'level': 1, 'lat': 181, 'lon': 360},
        decode_times=False,
    )
    ds = ds.drop_vars('step', errors='ignore')
    ds = xr.decode_cf(ds)
    return standardize_da(pick_data_var(ds, target))


def _slice_bounds(band: slice) -> tuple[float, float]:
    lower = float(min(band.start, band.stop))
    upper = float(max(band.start, band.stop))
    return lower, upper


def assert_coord_within(da: xr.DataArray | xr.Dataset, coord: str, band: slice, label: str) -> None:
    if coord not in da.coords:
        raise AssertionError(f'{label}: missing {coord!r} coordinate')
    values = np.asarray(da[coord].values)
    if values.size == 0:
        raise AssertionError(f'{label}: empty {coord!r} coordinate after domain selection')
    lower, upper = _slice_bounds(band)
    coord_min = float(np.nanmin(values))
    coord_max = float(np.nanmax(values))
    if coord_min < lower - 1e-6 or coord_max > upper + 1e-6:
        raise AssertionError(
            f'{label}: {coord} range {coord_min:.3f}..{coord_max:.3f} is outside {lower:.3f}..{upper:.3f}'
        )


def select_indonesia_domain(da: xr.DataArray, label: str) -> xr.DataArray:
    da = standardize_da(da)
    if 'lat' not in da.coords or 'lon' not in da.coords:
        raise ValueError(f'{label}: expected lat and lon coordinates')

    subset = da.sel(lat=LAT_BAND, lon=LON_BAND)
    if subset.sizes.get('lat', 0) == 0 or subset.sizes.get('lon', 0) == 0:
        raise ValueError(f'{label}: empty Indonesia domain selection for lat={LAT_BAND}, lon={LON_BAND}')

    assert_coord_within(subset, 'lat', LAT_BAND, label)
    assert_coord_within(subset, 'lon', LON_BAND, label)
    return subset


def assert_section_domain(da: xr.DataArray, x_dim: str, label: str) -> None:
    if x_dim == 'lon':
        assert_coord_within(da, 'lon', LON_BAND, label)
    elif x_dim == 'lat':
        assert_coord_within(da, 'lat', LAT_BAND, label)
    else:
        raise ValueError(f'Unsupported section dimension: {x_dim}')


def print_djf_completeness(name: str, da: xr.DataArray) -> None:
    time = da.sel(time=slice(TIME_START, TIME_END)).time
    time = time.sel(time=time.dt.month.isin((12, 1, 2)))
    years = (time.dt.year + xr.where(time.dt.month == 12, 1, 0)).values.astype(int)
    counts = pd.Series(1, index=pd.Index(years, name='djf_year')).groupby(level='djf_year').sum()
    counts = counts.reindex(DJF_YEARS, fill_value=0)
    incomplete = counts[counts < 3]
    p2_incomplete = incomplete[incomplete.index.isin(P2_YEARS)]

    print(f'{name} DJF month counts: min={int(counts.min())}, max={int(counts.max())}')
    if len(incomplete):
        print(f'WARNING {name}: incomplete DJF years (<3 months): {list(incomplete.index.astype(int))}')
    if len(p2_incomplete):
        print(f'WARNING {name}: incomplete P2 DJF years (<3 months): {list(p2_incomplete.index.astype(int))}')
    if 2025 in counts.index:
        print(f'{name} DJF 2025 month count: {int(counts.loc[2025])}')


def build_djf_means(da: xr.DataArray) -> xr.DataArray:
    da = da.sel(time=slice(TIME_START, TIME_END))
    da = da.sel(time=da.time.dt.month.isin((12, 1, 2)))
    djf_year = xr.where(da.time.dt.month == 12, da.time.dt.year + 1, da.time.dt.year)
    da = da.assign_coords(djf_year=('time', djf_year.data))
    da = da.sel(time=da.djf_year.isin(DJF_YEARS))
    return da.groupby('djf_year').mean('time')


def load_nino34_djf(path: Path) -> xr.DataArray:
    df = pd.read_csv(path, parse_dates=['Date'])
    value_col = next(col for col in df.columns if col != 'Date')

    series = pd.to_numeric(df[value_col], errors='coerce').replace([-9999.0, -99.99], np.nan)
    series = pd.Series(series.to_numpy(), index=df['Date'], name='nino34').sort_index()
    series = series.loc[TIME_START:TIME_END]
    series = series[series.index.month.isin((12, 1, 2))].copy()
    series.index = pd.Index(
        series.index.year + (series.index.month == 12).astype(int),
        name='djf_year',
    )

    djf = series.groupby(level='djf_year').mean()
    djf = djf.loc[DJF_YEARS[0] : DJF_YEARS[-1]]

    return xr.DataArray(
        djf.to_numpy(dtype=np.float32),
        coords={'djf_year': djf.index.to_numpy(dtype=np.int32)},
        dims=('djf_year',),
        name='nino34',
    )


def corr_for_period(field: xr.DataArray, nino34: xr.DataArray, years: np.ndarray, x_dim: str) -> xr.DataArray:
    field_sel = field.sel(djf_year=years)
    nino_sel = nino34.sel(djf_year=years)
    corr = xr.corr(field_sel, nino_sel, dim='djf_year')
    corr = corr.transpose('level', x_dim).astype('float32')
    assert_section_domain(corr, x_dim, f'corr {field.name or "field"}')
    return corr


def build_section_dataset(
    section_name: str,
    horiz_name: str,
    q_corr_p1: xr.DataArray,
    q_corr_p2: xr.DataArray,
    horiz_corr_p1: xr.DataArray,
    horiz_corr_p2: xr.DataArray,
    omega_corr_p1: xr.DataArray,
    omega_corr_p2: xr.DataArray,
) -> xr.Dataset:
    ds = xr.Dataset(
        {
            'q_corr_p1': q_corr_p1,
            'q_corr_p2': q_corr_p2,
            'q_corr_delta': (q_corr_p2 - q_corr_p1).astype('float32'),
            f'{horiz_name}_corr_p1': horiz_corr_p1,
            f'{horiz_name}_corr_p2': horiz_corr_p2,
            f'{horiz_name}_corr_delta': (horiz_corr_p2 - horiz_corr_p1).astype('float32'),
            'omega_corr_p1': omega_corr_p1,
            'omega_corr_p2': omega_corr_p2,
            'omega_corr_delta': (omega_corr_p2 - omega_corr_p1).astype('float32'),
        }
    )

    ds.attrs.update(
        {
            'title': f'DJF Niño3.4 correlation cache for {section_name} cross-section',
            'section': section_name,
            'period': '1981-2025',
            'split_p1': '1981-2006',
            'split_p2': '2007-2025',
            'djf_definition': 'December belongs to the following DJF year; DJF means are Dec-Jan-Feb averages.',
            'target_domain': '95E-125E, 10S-5N',
            'note': 'Cross-sections are restricted to the Indonesia-region domain before averaging. Quiver variables are correlations, not physical wind vectors.',
        }
    )
    return ds


def print_field_stats(name: str, da: xr.DataArray) -> None:
    print(f'\n{name}')
    print('  dims:', da.dims)
    print('  shape:', tuple(da.shape))
    for coord in ('lat', 'lon', 'level'):
        if coord in da.coords:
            values = np.asarray(da[coord].values)
            print(f'  {coord} range: {float(np.nanmin(values)):.3f}..{float(np.nanmax(values)):.3f}')

    valid_count = int(da.notnull().sum().compute().item())
    total_count = int(da.size)
    valid_fraction = valid_count / total_count if total_count else np.nan
    print(f'  valid fraction: {valid_fraction:.3f}')
    if valid_count == 0:
        print(f'  WARNING {name}: all values are NaN')
        return

    print(f'  min/max: {float(da.min(skipna=True).compute().item()):.4f} / {float(da.max(skipna=True).compute().item()):.4f}')


def print_dataset_diagnostics(label: str, ds: xr.Dataset) -> None:
    print(f'\n=== {label} cache diagnostics ===')
    print('dataset dims:', dict(ds.sizes))
    if 'lon' in ds.coords:
        assert_coord_within(ds, 'lon', LON_BAND, f'{label} cache')
    if 'lat' in ds.coords:
        assert_coord_within(ds, 'lat', LAT_BAND, f'{label} cache')
    for name in ds.data_vars:
        print_field_stats(name, ds[name])


def write_cache(ds: xr.Dataset, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and not OVERWRITE:
        print(f'WARNING cache exists and OVERWRITE=False; reusing old cache: {path}')
        return
    ds.to_netcdf(path)
    print(f'Wrote cache: {path}')


# 2. Open Data

In [ ]:
q = load_era5_field(Q_PATH, 'q')
u = load_era5_field(UV_PATH, 'u')
v = load_era5_field(UV_PATH, 'v')
omega = load_era5_field(W_PATH, 'w')
nino34 = load_nino34_djf(NINO34_PATH)

print('q dims:', dict(q.sizes))
print('u dims:', dict(u.sizes))
print('v dims:', dict(v.sizes))
print('omega dims:', dict(omega.sizes))
print('nino34 dims:', dict(nino34.sizes))

# 3. Compute DJF Correlations

In [ ]:
q_domain = select_indonesia_domain(q, 'q')
u_domain = select_indonesia_domain(u, 'u')
v_domain = select_indonesia_domain(v, 'v')
omega_domain = select_indonesia_domain(omega, 'omega')

for name, da in [('q', q_domain), ('u', u_domain), ('v', v_domain), ('omega', omega_domain)]:
    print_djf_completeness(name, da)

q_lon = build_djf_means(q_domain).mean('lat')
u_lon = build_djf_means(u_domain).mean('lat')
omega_lon = build_djf_means(omega_domain).mean('lat')

q_lat = build_djf_means(q_domain).mean('lon')
v_lat = build_djf_means(v_domain).mean('lon')
omega_lat = build_djf_means(omega_domain).mean('lon')

for label, da in [('q_lon', q_lon), ('u_lon', u_lon), ('omega_lon', omega_lon)]:
    assert_section_domain(da, 'lon', label)
for label, da in [('q_lat', q_lat), ('v_lat', v_lat), ('omega_lat', omega_lat)]:
    assert_section_domain(da, 'lat', label)

lon_q_p1 = corr_for_period(q_lon, nino34, P1_YEARS, 'lon')
lon_q_p2 = corr_for_period(q_lon, nino34, P2_YEARS, 'lon')
lon_u_p1 = corr_for_period(u_lon, nino34, P1_YEARS, 'lon')
lon_u_p2 = corr_for_period(u_lon, nino34, P2_YEARS, 'lon')
lon_omega_p1 = corr_for_period(omega_lon, nino34, P1_YEARS, 'lon')
lon_omega_p2 = corr_for_period(omega_lon, nino34, P2_YEARS, 'lon')

lat_q_p1 = corr_for_period(q_lat, nino34, P1_YEARS, 'lat')
lat_q_p2 = corr_for_period(q_lat, nino34, P2_YEARS, 'lat')
lat_v_p1 = corr_for_period(v_lat, nino34, P1_YEARS, 'lat')
lat_v_p2 = corr_for_period(v_lat, nino34, P2_YEARS, 'lat')
lat_omega_p1 = corr_for_period(omega_lat, nino34, P1_YEARS, 'lat')
lat_omega_p2 = corr_for_period(omega_lat, nino34, P2_YEARS, 'lat')

lon_ds = build_section_dataset(
    'longitude-pressure',
    'u',
    lon_q_p1,
    lon_q_p2,
    lon_u_p1,
    lon_u_p2,
    lon_omega_p1,
    lon_omega_p2,
)
lat_ds = build_section_dataset(
    'latitude-pressure',
    'v',
    lat_q_p1,
    lat_q_p2,
    lat_v_p1,
    lat_v_p2,
    lat_omega_p1,
    lat_omega_p2,
)

print('lon q/u/omega shapes:', lon_q_p1.shape, lon_u_p1.shape, lon_omega_p1.shape)
print('lat q/v/omega shapes:', lat_q_p1.shape, lat_v_p1.shape, lat_omega_p1.shape)


# 4. Save Cache

In [ ]:
print_dataset_diagnostics('longitude-pressure', lon_ds)
print_dataset_diagnostics('latitude-pressure', lat_ds)

write_cache(lon_ds, LON_CACHE_PATH)
write_cache(lat_ds, LAT_CACHE_PATH)


# 5. Revision Summary

- Fixed longitude-pressure and latitude-pressure section construction so both latitude and longitude are subset to `95E-125E, 10S-5N` before averaging.
- Added coordinate normalization, DJF completeness diagnostics, cache field statistics, domain assertions, and all-NaN warnings.
- `OVERWRITE=True` is intentionally enabled for validation so old global-domain caches are not reused.
- Omega P2 NaN status is explicitly reported by the cache diagnostics when this notebook is run.
